In [ ]:
# @title Cell 1: Install system packages & Python libraries
%cd /content
!apt-get update -y && apt-get install -y ffmpeg
!pip install -q youtube-dl numpy==1.23.5 librosa==0.8.1 \
    opencv-python==4.10.0.84 moviepy face_recognition dlib


In [ ]:
# @title Cell 2: Clone Wav2Lip & download model
%cd /content
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!pip install -q -r requirements.txt
!mkdir -p checkpoints
!gdown --id 1_OvqStxNxLc7bXzlaVG5sz695p-FVfYY \
      -O checkpoints/wav2lip.pth


In [ ]:
# @title Cell 3: Upload Inputs & Mute Original Video (with re‑prompt for missing audio)
from google.colab import files
import shutil, os, subprocess

# Step 1: Upload video + (optionally) audio
print("📤 Upload 1 video + your character .wav/.mp3 files now →")
uploaded = files.upload()

# Separate video vs audio
video_orig = next(f for f in uploaded
                  if f.lower().endswith(('.mp4','.mov','.avi','.mkv')))
audio_files = [f for f in uploaded
               if f.lower().endswith(('.wav','.mp3'))]

# Step 2: If no audio detected, re-prompt
if not audio_files:
    print("\n⚠️ No audio files detected. Please upload your character .wav/.mp3 files now →")
    more = files.upload()
    extra = [f for f in more if f.lower().endswith(('.wav','.mp3'))]
    audio_files += extra
    if not audio_files:
        raise RuntimeError("No audio files provided. Please restart this cell and upload at least one .wav/.mp3.")

# Step 3: Move into inputs/ & normalize names
os.makedirs('inputs', exist_ok=True)
shutil.move(video_orig,          'inputs/input_video_orig.mp4')
for a in audio_files:
    shutil.move(a, f'inputs/{a}')

# Step 4: Mute the original video → silent version for processing
subprocess.run([
    'ffmpeg', '-y',
    '-i', 'inputs/input_video_orig.mp4',
    '-c', 'copy',
    '-an',
    'inputs/input_video_silent.mp4'
], check=True)

print("\n✅ Muted video → inputs/input_video_silent.mp4")
print("🎵 Audio files →", [os.path.basename(a) for a in audio_files])


In [ ]:
# @title Cell 4A: Detect & Display Faces
import cv2
import face_recognition
import pickle
import os
from matplotlib import pyplot as plt

# Load first frame
cap = cv2.VideoCapture('inputs/input_video_silent.mp4')
ret, frame = cap.read()
cap.release()
if not ret:
    raise RuntimeError("Cannot read first frame")

# Detect faces
rgb = frame[:, :, ::-1]
boxes = face_recognition.face_locations(rgb, model='hog')  # [(top,right,bottom,left),...]

# Display grid
n = len(boxes)
if n == 0:
    raise RuntimeError("No faces detected!")
cols = min(4, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
axes = axes.flatten()
for i, (top, right, bottom, left) in enumerate(boxes):
    face = frame[top:bottom, left:right]
    axes[i].imshow(face[:, :, ::-1])
    axes[i].set_title(f"#{i}")
    axes[i].axis('off')
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

# Save for the next step
os.makedirs('faces', exist_ok=True)
with open('faces/boxes.pkl', 'wb') as f:
    pickle.dump({'frame': frame, 'boxes': boxes}, f)

print(f"Detected {n} faces. Now run Cell 4B to name them.")


In [ ]:
# @title Cell 4B: Name Faces & Save Initial BBoxes
import pickle, os

# Load frame & boxes from Cell 4A
with open('faces/boxes.pkl','rb') as f:
    data   = pickle.load(f)
boxes   = data['boxes']   # [(top, right, bottom, left), ...]

# 1) Prompt for names
names = {}
print("Detected face indices:", list(range(len(boxes))))
for i in range(len(boxes)):
    name = input(f"Enter name for face #{i} (blank to skip): ").strip()
    if name:
        names[name] = boxes[i]

# 2) Save mapping name → bbox
os.makedirs('faces', exist_ok=True)
with open('faces/named_boxes.pkl','wb') as f:
    pickle.dump(names, f)

print("Saved named boxes for:", list(names.keys()))


In [ ]:
# @title Cell 5: Smooth, Continuous Masking via Detection + IoU Tracking
import pickle, os, cv2
import numpy as np
import face_recognition

def iou(boxA, boxB):
    tA, rA, bA, lA = boxA
    tB, rB, bB, lB = boxB
    xA, yA = max(lA, lB), max(tA, tB)
    xB, yB = min(rA, rB), min(bA, bB)
    if xB <= xA or yB <= yA: return 0.0
    inter = (xB - xA) * (yB - yA)
    areaA = (rA - lA) * (bA - tA)
    areaB = (rB - lB) * (bB - tB)
    return inter / float(areaA + areaB - inter)

# Load named initial boxes
with open('faces/named_boxes.pkl','rb') as f:
    named_boxes = pickle.load(f)  # { name: (top,right,bottom,left) }

# Video I/O
inp = 'inputs/input_video_silent.mp4'
cap = cv2.VideoCapture(inp)
fps = cap.get(cv2.CAP_PROP_FPS)
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

os.makedirs('masks', exist_ok=True)
writers = {
    name: cv2.VideoWriter(
        f'masks/{name}.mp4',
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps, (W, H)
    )
    for name in named_boxes
}

# State: last_boxes + smoothed_boxes
last_boxes     = named_boxes.copy()
smoothed_boxes = named_boxes.copy()
alpha = 0.3   # smoothing factor: 0 = no update, 1 = no smoothing

while True:
    ret, frame = cap.read()
    if not ret:
        break
    rgb = frame[:, :, ::-1]

    # 1) Detect all faces
    dets = face_recognition.face_locations(rgb, model='hog')

    # 2) For each named face, pick best detection by IoU
    new_boxes = {}
    for name, prev in last_boxes.items():
        best_i, best_box = 0, None
        for d in dets:
            score = iou(prev, d)
            if score > best_i:
                best_i, best_box = score, d
        # fallback: if no good overlap, keep previous
        if best_i > 0.2:
            chosen = best_box
        else:
            chosen = prev
        new_boxes[name] = chosen

    # 3) Smooth the box coordinates
    for name, box in new_boxes.items():
        prev = smoothed_boxes[name]
        smoothed = tuple(
            int(alpha*new + (1-alpha)*old)
            for new, old in zip(box, prev)
        )
        smoothed_boxes[name] = smoothed

    # 4) Mask & write each frame
    for name, box in smoothed_boxes.items():
        t, r, b, l = box
        mask = np.zeros((H, W), dtype=np.uint8)
        # clamp coords
        t, l = max(0, t), max(0, l)
        b, r = min(H, b), min(W, r)
        mask[t:b, l:r] = 255
        masked = cv2.bitwise_and(frame, frame, mask=mask)
        writers[name].write(masked)

    # update for next iteration
    last_boxes = new_boxes

cap.release()
for w in writers.values():
    w.release()

print("✅ Smoothed, continuous masks at masks/<name>.mp4")


In [ ]:
# @title Cell 6: Patch inference.py to force GPU & Run Wav2Lip Inference
import os, glob

wav2lip_dir = '/content/Wav2Lip'
inf_py      = os.path.join(wav2lip_dir, 'inference.py')

# 0) Patch to force GPU
patch_gpu = [
    # make only GPU 0 visible
    'import os; os.environ["CUDA_VISIBLE_DEVICES"]="0"\n',
    # after torch import, force device
    'import torch; device = torch.device("cuda")\n'
]

# Apply the first patch before any other imports
with open(inf_py, 'r+') as f:
    text = f.read()
    if 'CUDA_VISIBLE_DEVICES' not in text:
        f.seek(0,0)
        f.write(patch_gpu[0] + text)
        print("[PATCHED] inference.py: set CUDA_VISIBLE_DEVICES")

# Now insert device override right after the torch import
lines = open(inf_py).read().splitlines()
for i, line in enumerate(lines):
    if 'import torch' in line:
        if 'device =' not in lines[i+1]:
            lines.insert(i+1, patch_gpu[1])
            print("[PATCHED] inference.py: override device to cuda")
        break
with open(inf_py, 'w') as f:
    f.write('\n'.join(lines))

# 1) Locate masks
mask_paths = glob.glob('/content/masks/*.mp4') + glob.glob(f'{wav2lip_dir}/masks/*.mp4')
print("🔍 Masks found:", mask_paths)

# 2) Ensure outputs dir
out_dir = '/content/outputs'
os.makedirs(out_dir, exist_ok=True)

# 3) Run inference using !python
for mask in mask_paths:
    name = os.path.splitext(os.path.basename(mask))[0]
    # find audio
    aud = None
    for d in ['/content/inputs', f'{wav2lip_dir}/inputs']:
        for ext in ('.wav','.mp3','.flac','.aac'):
            p = f'{d}/{name}{ext}'
            if os.path.exists(p):
                aud = p; break
        if aud: break
    if not aud:
        print(f"⚠️ No audio for {name}, skipping.")
        continue

    outp = f'{out_dir}/{name}_synced.mp4'
    print(f"\n▶️ {name}: mask={mask}, audio={aud}, out={outp}")
    get_ipython().system(f"python {inf_py} \
        --checkpoint_path {wav2lip_dir}/checkpoints/wav2lip.pth \
        --face {mask} \
        --audio {aud} \
        --outfile {outp}")

print("\n✅ Inference done. Check /content/outputs for synced videos.")



In [ ]:
# @title Cell 7: Combine the Videos
import os, glob, cv2, numpy as np
from moviepy.editor import VideoFileClip, CompositeVideoClip, AudioFileClip
from IPython.display import HTML

# 1) Load base
cands = glob.glob('/content/inputs/input_video_silent.*') + glob.glob('/content/Wav2Lip/inputs/input_video_silent.*')
base = VideoFileClip(cands[0])

# 2) Face clips
paths = sorted(glob.glob('/content/outputs/*_synced.mp4'))
if not paths: raise RuntimeError("No face clips")

# 3) Mask cleaner
def refine_mask(clip):
    mask = clip.to_mask()
    def clean_frame(m):
        m_bin = (m > 0.3).astype(np.uint8) * 255
        k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(21,21))
        m_bin = cv2.morphologyEx(m_bin, cv2.MORPH_CLOSE, k1)
        k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(11,11))
        m_bin = cv2.morphologyEx(m_bin, cv2.MORPH_OPEN, k2)
        k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))
        m_bin = cv2.erode(m_bin, k3, iterations=1)
        m_bin = cv2.GaussianBlur(m_bin, (15,15), 0)
        return (m_bin/255.0).astype(np.float32)
    return clip.set_mask(mask.fl_image(clean_frame))

# 4) Build layers
layers = [base]
for p in paths:
    clip = VideoFileClip(p)
    layers.append(refine_mask(clip))

# 5) Composite + audio (use your mixed audio if available)
final = CompositeVideoClip(layers, size=base.size)
# final = final.set_audio(AudioFileClip('/content/final_audio_mix.mp3'))

# 6) Export & preview
out = '/content/final_composite.mp4'
final.write_videofile(out, codec='libx264', audio_codec='aac')
display(HTML(f"""<video width=512 controls>
  <source src="{out}" type="video/mp4">
</video>"""))
